<a href="https://colab.research.google.com/github/emily-escudero/Analitica-Educacion-rural/blob/main/Entrega2EducacionRural.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests

# Definir la API URL
url = "https://www.datos.gov.co/api/v3/views/ji8i-4anb/query.json"

# solicitud a la API
respuesta_api = requests.get(url)
respuesta_api.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
datos = respuesta_api.json()


df = pd.DataFrame(datos)
display(df.head())

,:id,:version,:created_at,:updated_at,ano,c_digo_departamento,departamento,poblacion_5_16,tasa_matriculacion_5_16,cobertura_neta,...,reprobacion,reprobacion_transicion,reprobacion_primaria,reprobacion_secundaria,reprobacion_media,repitencia,repitencia_transicion,repitencia_primaria,repitencia_secundaria,repitencia_media
0,row-xccu-54yf.9ie2,rv-69kt.ykig.crwc,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,5,Antioquia,1288473,94.01,93.85,...,2.06,0.07,94.56,2.54,2.96,4.25,0.07,4.56,5.27,1.68
1,row-755p~4uyd-2k3w,rv-ar5t~pmwz_z3hy,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,8,Atlántico,523935,99.32,99.05,...,0.54,0.12,96.49,0.67,0.75,1.82,0.12,1.77,2.18,0.88
2,row-rb29.mx9s.a7ax,rv-atze~ze7f~jiue,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,11,"Bogotá, D.C.",1479334,90.7,90.29,...,0,0,94.69,0,0,3.23,0,2.3,5.11,2.57
3,row-gavf~qv4q~dhdy,rv-9idw~dvm2~k7me,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,13,Bolívar,496676,91.57,91.4,...,2.1,0.46,95.48,2.75,3.67,4.43,0.46,4.44,5.37,2.28
4,row-zz5g~58au~qymy,rv-68rp.uscp_a52j,2024-01-30T16:43:35.793Z,2024-01-30T16:43:35.793Z,2011,15,Boyacá,300501,86.16,86.11,...,2.73,0.17,96.1,4.31,3.26,2.62,0.17,1.9,4.19,1.55


In [ ]:
# Definir columnas
columnas_desercion = [
    'departamento',
    'ano',
    'desercion',
    'desercion_transicion',
    'desercion_primaria',
    'desercion_secundaria',
    'desercion_media'
]

# Seleccionamos las columnas necesarias
datos_desercion = df[df.columns.intersection(columnas_desercion)].copy()

# Convertir la columna año a formato numerico
datos_desercion['ano'] = pd.to_numeric(datos_desercion['ano'], errors='coerce')

# Filtrar la informacion entre el 2021 y 2024
df_deserciones_2021_2024 = datos_desercion[
    (datos_desercion['ano'] >= 2021) & (datos_desercion['ano'] <= 2024)
].copy()

# Display the first few rows of the filtered DataFrame
display(df_deserciones_2021_2024.head())

,ano,departamento,desercion,desercion_transicion,desercion_primaria,desercion_secundaria,desercion_media
330,2021,Antioquia,4.81,3.59,4.27,5.90,4.13
331,2021,Atlántico,1.67,2.15,1.75,1.68,1.03
332,2021,"Bogotá, D,C,",1.29,1.07,1.08,1.35,1.88
333,2021,Bolívar,3.69,4.13,3.33,4.27,3.13
334,2021,Boyacá,2.97,2.75,2.13,3.63,3.69


In [ ]:
import pandas as pd
import requests

# URL del archivo excel en github
excel_url = "https://github.com/emily-escudero/Analitica-Educacion-rural/raw/main/anex-pobrezadepartamental.xlsx"

# Nombre del archivo
pobreza_departamental = "anex-pobrezadepartamental.xlsx"

# Descargar el archivo excel
print(f"Downloading {excel_url}...")
response = requests.get(excel_url)
response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

with open(pobreza_departamental, 'wb') as f:
    f.write(response.content)

print(f"Archivo descargado {pobreza_departamental}")

Archivo descargado anex-pobrezadepartamental.xlsx


### Preparación de datos de pobreza monetaria

En este paso se extraen los datos de la hoja Pobreza Monetaria Act.Met. del archivo Excel. Los nombres de los departamentos se encuentran en la columna A, desde la fila 12 hasta la 35, mientras que los años están ubicados en la fila 11, desde las columnas B hasta F. Posteriormente, la información se organiza para facilitar su análisis y filtrado.

In [ ]:
# Load the 'Pobreza Monetaria Act.Met.' sheet
# We need to read the specific range for departments and lista_años.
# We'll read the data from the relevant cells and then process it.

# Leer la fila de loa lista_años
tabla_años = pd.read_excel(
    pobreza_departamental,
    sheet_name='Pobreza Monetaria Act.Met.',
    header=None, # No default header
    skiprows=10, # Skip to row 11 (0-indexed)
    nrows=1,     # Read only one row for lista_años
    usecols='B:F' # Use columns B to F for lista_años
)

# Extraer los lista_años a la hoja de escel
lista_años = [int(col) for col in tabla_años.iloc[0].values]

# Leer los datos de pobreza monetaria por departamento
monetaria_df = pd.read_excel(
    pobreza_departamental,
    sheet_name='Pobreza Monetaria Act.Met.',
    header=None, # No default header
    skiprows=11, # Skip to row 12 (0-indexed)
    nrows=24,    # From row 12 to 35 (35-12+1 = 24 rows)
    usecols='A:F' # Use columns A to F
)

# Asignar nombres a las columnas
monetaria_df.columns = ['departamento'] + lista_años

# convertir la tabla de formato ancho a formato largo
df_monetaria_melted = monetaria_df.melt(
    id_vars=['departamento'],
    var_name='ano',
    value_name='pobreza_monetaria'
)

# Filtrar la lista de años del 2021 al 2024
df_pobreza_monetaria = df_monetaria_melted[
    (df_monetaria_melted['ano'] >= 2021) & (df_monetaria_melted['ano'] <= 2024)
].copy()

# Display the first few rows
print("Pobreza Monetaria (2021-2024):")
display(df_pobreza_monetaria.head())

Pobreza Monetaria (2021-2024):


,departamento,ano,pobreza_monetaria
0,Antioquia,2021,32.8
1,Atlántico,2021,42.1
2,Bogotá D.C.,2021,30.5
3,Bolívar,2021,54.0
4,Boyacá,2021,41.8


### Preparacion de datos de pobreaza extrema

A continuación, se procesan los datos correspondientes a la hoja Pobreza Extrema Act.Met.. En esta hoja, los nombres de los departamentos se encuentran desde la celda A15 hasta A38, mientras que los años están ubicados en la fila 14, desde las columnas B hasta F.

In [ ]:

# Leer la fila donde se encuntran los años
tabla_años_extrema = pd.read_excel(
    pobreza_departamental,
    sheet_name='Pobreza Extrema Act.Met.',
    header=None,
    skiprows=13,
    nrows=1,
    usecols='B:F'
)

# Extraer años de tabla_años_extrema
lista_años_extrema = [int(col) for col in tabla_años_extrema.iloc[0].values]

# Leer los datos de pobreza extrema por departamento
datos_pobreza_extrema = pd.read_excel(
    pobreza_departamental,
    sheet_name='Pobreza Extrema Act.Met.',
    header=None,
    skiprows=14,
    nrows=24,
    usecols='A:F'
)

# Asignar nombres a las columnas
datos_pobreza_extrema.columns = ['departamento'] + lista_años_extrema

# Convertir la tabla de formato ancho a formato largo
df_extrema_melted = datos_pobreza_extrema.melt(
    id_vars=['departamento'],
    var_name='ano',
    value_name='pobreza_extrema'
)

# Filtrar años del 2021 al 2024
df_pobreza_extrema = df_extrema_melted[
    (df_extrema_melted['ano'] >= 2021) & (df_extrema_melted['ano'] <= 2024)
].copy()

# Mostrar primeras filas
print("Pobreza Extrema (2021-2024):")
display(df_pobreza_extrema.head())

Pobreza Extrema (2021-2024):


,departamento,ano,pobreza_extrema
0,Antioquia,2021,9.2
1,Atlántico,2021,12.3
2,Bogotá D.C.,2021,8.4
3,Bolívar,2021,18.8
4,Boyacá,2021,16.4


### Union base de datos

En este paso se integran los datos de pobreza monetaria y pobreza extrema en una sola base de datos, de manera que cada registro contenga la información correspondiente al mismo departamento y año.

In [ ]:
# Unir datos de pobreza monetaria a pobreza extrema
df_pobreza_completos = pd.merge(
    df_pobreza_monetaria,
    df_pobreza_extrema,
    on=['departamento', 'ano'],
    how='inner'
)

# Mostrar base de datos final
print("Final DataFrame with Pobreza Monetaria and Pobreza Extrema (2021-2024):")
display(df_pobreza_completos)

Final DataFrame with Pobreza Monetaria and Pobreza Extrema (2021-2024):


,departamento,ano,pobreza_monetaria,pobreza_extrema
0,Antioquia,2021,32.8,9.2
1,Atlántico,2021,42.1,12.3
2,Bogotá D.C.,2021,30.5,8.4
3,Bolívar,2021,54.0,18.8
4,Boyacá,2021,41.8,16.4
...,...,...,...,...
91,Risaralda,2024,23.8,5.2
92,Santander,2024,27.4,7.7
93,Sucre,2024,57.5,23.7
94,Tolima,2024,35.6,12.2


In [ ]:
import pandas as pd
import requests

# Url del archivo de educacion formal
educacion_formal = "https://github.com/emily-escudero/Analitica-Educacion-rural/raw/main/anex-educacionformal.xlsx"

# Nombre con el que guardo el archivo
ruta_educacion_formal = "anex-educacionformal.xlsx"

# Descargar archivo de excel
print(f"Downloading {educacion_formal}...")
respuesta_descarga = requests.get(educacion_formal)
respuesta_descarga.raise_for_status() # Raise an HTTPError for bad respuesta_descargas (4xx or 5xx)

with open(ruta_educacion_formal, 'wb') as f:
    f.write(respuesta_descarga.content)

print(f"Se dercargo el archivo {ruta_educacion_formal}")

Se dercargo el archivo anex-educacionformal.xlsx


### Preparación de Datos de Sedes Educativas

Ahora vamos a cargar los datos de la hoja `Sedes_nivel educativo_zona`. Según tus indicaciones:
*   Los departamentos están en la columna `A` desde la fila `19` hasta la `51`.
*   El número de sedes urbanas va desde la columna `B` hasta la `F` en la fila `17`.
*   El número de sedes rurales va desde la columna `G` hasta la `K` en la fila `17`.

In [ ]:

# 1. leer los nombres de las categorias de educacion
encabezados_educacion = pd.read_excel(
    ruta_educacion_formal,
    sheet_name='Sedes_nivel educativo_zona',
    header=None,
    skiprows=16,
    nrows=1,
    usecols='B:K'
)
nombres_educacion = encabezados_educacion.iloc[0].tolist()

# crear nombres para las columnas  urbanas y rurales
columnas_urbanas = [f'U_{name}' for name in nombres_educacion[:5]] # e.g., U_Preescolar
columnaas_rurales = [f'R_{name}' for name in nombres_educacion[5:]] # e.g., R_Preescolar
columnas_numericas = columnas_urbanas + columnaas_rurales


# 2. Leer los nombres de los departamentos
datos_departamentos = pd.read_excel(
    ruta_educacion_formal,
    sheet_name='Sedes_nivel educativo_zona',
    header=None,
    skiprows=18,
    nrows=33,
    usecols='A'
)
lista_departamentos = datos_departamentos.iloc[:, 0].tolist()

# 3. Leer los datos de sedes educativas urbanas y rurales
datos_sedes = pd.read_excel(
    ruta_educacion_formal,
    sheet_name='Sedes_nivel educativo_zona',
    header=None,
    skiprows=18,
    nrows=33,
    usecols='B:K'
)

# Asignar nombres a las columnas
datos_sedes.columns = columnas_numericas

#  DataFrame
df_instituciones_temp = pd.DataFrame(datos_sedes)
df_instituciones_temp.insert(0, 'departamento', lista_departamentos)

# Calcular el total de sedes educativas
df_instituciones_temp.iloc[:, 1:] = df_instituciones_temp.iloc[:, 1:].fillna(0)



df_instituciones = pd.DataFrame({
    'departamento': df_instituciones_temp['departamento'],
    'total_sedes_urbanas': df_instituciones_temp[columnas_urbanas].sum(axis=1),
    'total_sedes_rurales': df_instituciones_temp[columnaas_rurales].sum(axis=1)
})

df_instituciones['total_sedes'] = df_instituciones['total_sedes_urbanas'] + df_instituciones['total_sedes_rurales']

# Mostrar resultados
print("Total de Instituciones Urbanas y Rurales por Departamento:")
display(df_instituciones.head())
display(df_instituciones.tail())

Total de Instituciones Urbanas y Rurales por Departamento:


,departamento,total_sedes_urbanas,total_sedes_rurales,total_sedes
0,Amazonas,47,161,208
1,Antioquia,4126,9853,13979
2,Arauca,277,846,1123
3,"Archipiélago de San Andrés, Providencia y Sant...",46,35,81
4,Atlántico,3210,271,3481


,departamento,total_sedes_urbanas,total_sedes_rurales,total_sedes
28,Sucre,824,1573,2397
29,Tolima,1501,3488,4989
30,Valle del Cauca,4446,3065,7511
31,Vaupés,25,224,249
32,Vichada,48,462,510


In [ ]:
import pandas as pd
import requests

# URL del archivo excel en github
excel_url_terridata = "https://github.com/emily-escudero/Analitica-Educacion-rural/raw/main/TerriData_educacion.xlsx"

# Nombre del archivo
terridata_educacion = "TerriData_educacion.xlsx"

# Descargar el archivo excel
print(f"Downloading {excel_url_terridata}...")
response = requests.get(excel_url_terridata)
response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

with open(terridata_educacion, 'wb') as f:
    f.write(response.content)

print(f"Archivo descargado {terridata_educacion}")

Archivo descargado TerriData_educacion.xlsx


### Preparación de datos de analfabetismo rural

En este paso, extraeremos los datos de la tasa de analfabetismo rural del archivo `TerriData_educacion.xlsx`. Asumo que hay una hoja llamada 'Analfabetismo Rural' y que la estructura es similar a los archivos de pobreza previamente procesados, con los años en una fila de encabezado y los departamentos en la primera columna.

### Preparación de datos de analfabetismo rural por municipio y luego agregación por departamento

En este paso, extraeremos los datos de la tasa de analfabetismo rural, la entidad (municipio), el departamento, el dato numérico de la columna H y el año del archivo `TerriData_educacion.xlsx`. Luego, promediaremos estos datos por departamento para consolidar la información.

In [ ]:
# Leer los datos específicos de departamento, entidad, tasa de analfabetismo rural, dato numérico y año
df_analfabetismo_rural_entidades = pd.read_excel(
    terridata_educacion,
    sheet_name='Hoja01',
    header=None, # No hay encabezado en las filas que vamos a leer
    skiprows=374895, # Saltar hasta la fila 374896 (0-indexed)
    nrows=1104,    # Leer 375999 - 374896 + 1 = 1104 filas
    usecols='B,D,G,H,J' # Columnas B (departamento), D (entidad), G (tasa analfabetismo), H (dato numérico), J (año)
)

# Asignar nombres a las columnas
df_analfabetismo_rural_entidades.columns = [
    'departamento',
    'entidad',
    'tasa_analfabetismo_rural',
    'dato_numerico_h',
    'ano'
]

# Convertir la columna 'ano' a tipo entero, ignorando errores
df_analfabetismo_rural_entidades['ano'] = pd.to_numeric(
    df_analfabetismo_rural_entidades['ano'], errors='coerce'
).fillna(0).astype(int)

# Mostrar las primeras filas del DataFrame resultante
print("Datos de Analfabetismo Rural por Entidad, Departamento, Dato Numérico y Año:")
display(df_analfabetismo_rural_entidades.head())
print("Últimas filas de los datos de Analfabetismo Rural:")
display(df_analfabetismo_rural_entidades.tail())

Datos de Analfabetismo Rural por Entidad, Departamento, Dato Numérico y Año:


,departamento,entidad,tasa_analfabetismo_rural,dato_numerico_h,ano
0,Antioquia,Antioquia,Tasa de Analfabetismo Rural (Censo),"10,79",2018
1,Antioquia,Abejorral,Tasa de Analfabetismo Rural (Censo),"11,10",2018
2,Antioquia,Abriaquí,Tasa de Analfabetismo Rural (Censo),"8,41",2018
3,Antioquia,Alejandría,Tasa de Analfabetismo Rural (Censo),"12,06",2018
4,Antioquia,Amagá,Tasa de Analfabetismo Rural (Censo),"9,84",2018


Últimas filas de los datos de Analfabetismo Rural:


,departamento,entidad,tasa_analfabetismo_rural,dato_numerico_h,ano
1099,Vichada,Vichada,Tasa de Analfabetismo Rural (Censo),"14,82",2018
1100,Vichada,La Primavera,Tasa de Analfabetismo Rural (Censo),"10,54",2018
1101,Vichada,Santa Rosalía,Tasa de Analfabetismo Rural (Censo),"9,30",2018
1102,Vichada,Cumaribo,Tasa de Analfabetismo Rural (Censo),"14,62",2018
1103,Colombia,Colombia,Tasa de Analfabetismo Rural (Censo),"12,08",2018


### Agregación de datos de Analfabetismo Rural por Departamento

Ahora vamos a procesar `df_analfabetismo_rural_entidades` para tener una única fila por departamento, promediando las tasas de analfabetismo rural y los datos numéricos de la columna H.

In [ ]:
# Convertir 'tasa_analfabetismo_rural' y 'dato_numerico_h' a numérico
# Reemplazar comas por puntos para la conversión a float
# Usar pd.to_numeric con errors='coerce' para convertir valores no numéricos a NaN

df_analfabetismo_rural_entidades['tasa_analfabetismo_rural'] = (
    df_analfabetismo_rural_entidades['tasa_analfabetismo_rural']
    .astype(str) # Convert to string first to use .str.replace
    .str.replace(',', '.', regex=False)
)
df_analfabetismo_rural_entidades['tasa_analfabetismo_rural'] = pd.to_numeric(
    df_analfabetismo_rural_entidades['tasa_analfabetismo_rural'], errors='coerce'
)

df_analfabetismo_rural_entidades['dato_numerico_h'] = (
    df_analfabetismo_rural_entidades['dato_numerico_h']
    .astype(str) # Convert to string first to use .str.replace
    .str.replace(',', '.', regex=False)
)
df_analfabetismo_rural_entidades['dato_numerico_h'] = pd.to_numeric(
    df_analfabetismo_rural_entidades['dato_numerico_h'], errors='coerce'
)

# Agrupar por departamento y calcular el promedio de las columnas numéricas, incluyendo el año
df_analfabetismo_rural_promedio = df_analfabetismo_rural_entidades.groupby(['departamento', 'ano']).agg(
    tasa_analfabetismo_rural_promedio=('tasa_analfabetismo_rural', 'mean'),
    dato_numerico_h_promedio=('dato_numerico_h', 'mean')
).reset_index()

# Mostrar el DataFrame agregado
print("Datos de Analfabetismo Rural promediados por Departamento y Año:")
display(df_analfabetismo_rural_promedio.head())
print(f"Número de filas después de la agregación: {len(df_analfabetismo_rural_promedio)}")

Datos de Analfabetismo Rural promediados por Departamento y Año:


,departamento,ano,tasa_analfabetismo_rural_promedio,dato_numerico_h_promedio
0,Amazonas,2018,NaN,9.185000
1,Antioquia,2018,NaN,11.854000
2,Arauca,2018,NaN,11.045714
3,Atlántico,2018,NaN,15.493913
4,Bolívar,2018,NaN,17.293696


Número de filas después de la agregación: 33


### Nota sobre los datos de Analfabetismo Rural

Según los rangos de celdas proporcionados (desde la fila 374896 hasta la 375999), parece que la columna 'J' contiene el mismo año para todas las entradas en este segmento de datos. Por lo tanto, el DataFrame `df_analfabetismo_rural` contendrá datos de un único año para la tasa de analfabetismo rural, junto con la tasa de analfabetismo y el dato numérico adicional de la columna 'H'.

### Consolidación de Bases de Datos

Vamos a unir todas las bases de datos preparadas (`df_deserciones_2021_2024`, `df_pobreza_completos`, `df_instituciones`, y `df_analfabetismo_rural_promedio`) en un único DataFrame. Para ello, primero normalizaremos los nombres de los departamentos para asegurar una unión correcta, ya que pueden haber inconsistencias (tildes, mayúsculas/minúsculas, etc.). Luego, como la información de las sedes educativas (`df_instituciones`) solo está disponible para un año, la replicaremos para cada año del rango 2021-2024. Finalmente, realizaremos las uniones secuenciales.

### Asegurar la Definición de DataFrames Previos

Debido a un error previo donde `df_analfabetismo_rural_promedio` no estaba definido al momento de la normalización, se añade este paso explícito para asegurar que tanto `df_analfabetismo_rural_entidades` como `df_analfabetismo_rural_promedio` estén correctamente cargados y procesados en el entorno. Esto es crucial antes de proceder con la normalización de nombres de departamentos y las uniones finales.

In [46]:
import pandas as pd

# Re-crear df_analfabetismo_rural_entidades (contenido de la celda 64f67fe9)
df_analfabetismo_rural_entidades = pd.read_excel(
    terridata_educacion,
    sheet_name='Hoja01',
    header=None,
    skiprows=374895,
    nrows=1104,
    usecols='B,D,G,H,J'
)
df_analfabetismo_rural_entidades.columns = [
    'departamento',
    'entidad',
    'tasa_analfabetismo_rural',
    'dato_numerico_h',
    'ano'
]
df_analfabetismo_rural_entidades['ano'] = pd.to_numeric(
    df_analfabetismo_rural_entidades['ano'], errors='coerce'
).fillna(0).astype(int)

# Re-crear df_analfabetismo_rural_promedio (contenido de la celda 2732d3a0)
df_analfabetismo_rural_entidades['tasa_analfabetismo_rural'] = (
    df_analfabetismo_rural_entidades['tasa_analfabetismo_rural']
    .astype(str)
    .str.replace(',', '.', regex=False)
)
df_analfabetismo_rural_entidades['tasa_analfabetismo_rural'] = pd.to_numeric(
    df_analfabetismo_rural_entidades['tasa_analfabetismo_rural'], errors='coerce'
)
df_analfabetismo_rural_entidades['dato_numerico_h'] = (
    df_analfabetismo_rural_entidades['dato_numerico_h']
    .astype(str)
    .str.replace(',', '.', regex=False)
)
df_analfabetismo_rural_entidades['dato_numerico_h'] = pd.to_numeric(
    df_analfabetismo_rural_entidades['dato_numerico_h'], errors='coerce'
)
df_analfabetismo_rural_promedio = df_analfabetismo_rural_entidades.groupby(['departamento', 'ano']).agg(
    tasa_analfabetismo_rural_promedio=('tasa_analfabetismo_rural', 'mean'),
    dato_numerico_h_promedio=('dato_numerico_h', 'mean')
).reset_index()

print("df_analfabetismo_rural_entidades y df_analfabetismo_rural_promedio han sido re-definidos.")

df_analfabetismo_rural_entidades y df_analfabetismo_rural_promedio han sido re-definidos.


In [47]:
import unicodedata
import re

# Función para normalizar nombres de departamentos
def normalize_department_name(name):
    if not isinstance(name, str):
        return name
    # Convertir a minúsculas
    name = name.lower()
    # Eliminar tildes
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('utf-8')
    # Reemplazar 'bogota d.c.' y 'bogota, d,c,' por 'bogota dc'
    name = re.sub(r'bogota[,\\s]*d[.\\s]*c[.\\s]*', 'bogota dc', name)
    # Eliminar caracteres especiales y espacios extra
    name = re.sub(r'[^a-z0-9\\s]', '', name)
    # Reemplazar múltiples espacios por uno solo y limpiar espacios al inicio/final
    name = re.sub(r'\\s+', ' ', name).strip()
    return name

# Aplicar la normalización a todos los DataFrames que se van a unir
# Asumimos que los DataFrames df_deserciones_2021_2024, df_pobreza_completos,
# df_instituciones, y df_analfabetismo_rural_promedio ya han sido definidos en celdas anteriores.
df_deserciones_2021_2024['departamento'] = df_deserciones_2021_2024['departamento'].apply(normalize_department_name)
df_pobreza_completos['departamento'] = df_pobreza_completos['departamento'].apply(normalize_department_name)
df_instituciones['departamento'] = df_instituciones['departamento'].apply(normalize_department_name)
df_analfabetismo_rural_promedio['departamento'] = df_analfabetismo_rural_promedio['departamento'].apply(normalize_department_name)

print("Nombres de departamentos normalizados en todos los DataFrames.")

Nombres de departamentos normalizados en todos los DataFrames.


In [42]:
# Preparar df_instituciones para la unión anual
# Crear un DataFrame con los años 2021 a 2024
years_to_expand = pd.DataFrame({'ano': range(2021, 2025)})

# Crear una clave para unir que replicará df_instituciones para cada año
df_instituciones['key_merge'] = 1
years_to_expand['key_merge'] = 1

df_instituciones_expanded = pd.merge(df_instituciones, years_to_expand, on='key_merge').drop('key_merge', axis=1)

# Reordenar las columnas para tener 'departamento', 'ano' primero
df_instituciones_expanded = df_instituciones_expanded[['departamento', 'ano', 'total_sedes_urbanas', 'total_sedes_rurales', 'total_sedes']]

print("DataFrame de instituciones expandido por año (2021-2024):")
display(df_instituciones_expanded.head())
display(df_instituciones_expanded.tail())

DataFrame de instituciones expandido por año (2021-2024):


,departamento,ano,total_sedes_urbanas,total_sedes_rurales,total_sedes
0,Amazonas,2021,47,161,208
1,Amazonas,2022,47,161,208
2,Amazonas,2023,47,161,208
3,Amazonas,2024,47,161,208
4,Antioquia,2021,4126,9853,13979


,departamento,ano,total_sedes_urbanas,total_sedes_rurales,total_sedes
127,Vaupés,2024,25,224,249
128,Vichada,2021,48,462,510
129,Vichada,2022,48,462,510
130,Vichada,2023,48,462,510
131,Vichada,2024,48,462,510


### Nota importante: Ejecución secuencial de celdas

Para evitar `NameError`s y asegurar la correcta ejecución del código, es fundamental ejecutar todas las celdas previas de forma secuencial, especialmente aquellas que definen DataFrames (`df_deserciones_2021_2024`, `df_pobreza_completos`, `df_instituciones`, `df_analfabetismo_rural_promedio`). Si encuentras un `NameError` nuevamente, por favor, asegúrate de haber ejecutado todas las celdas desde el inicio del notebook.

In [48]:
# Realizar la primera unión: Deserciones y Pobreza
# Usamos 'outer' para mantener todas las combinaciones posibles y ver qué falta
df_final = pd.merge(
    df_deserciones_2021_2024,
    df_pobreza_completos,
    on=['departamento', 'ano'],
    how='left'  # Mantener todas las filas de deserciones
)

# Unir con los datos de instituciones (ya expandidos por año)
df_final = pd.merge(
    df_final,
    df_instituciones_expanded,
    on=['departamento', 'ano'],
    how='left' # Mantener todas las filas de la unión anterior
)

# Unir con los datos de analfabetismo
df_final = pd.merge(
    df_final,
    df_analfabetismo_rural_promedio,
    on=['departamento', 'ano'],
    how='left' # Mantener todas las filas de la unión anterior
)

# Mostrar el DataFrame final
print("DataFrame Final consolidado (primeras 5 filas):")
display(df_final.head())
print("DataFrame Final consolidado (últimas 5 filas):")
display(df_final.tail())
print(f"Dimensiones del DataFrame final: {df_final.shape}")
print(f"Columnas del DataFrame final: {df_final.columns.tolist()}")

DataFrame Final consolidado (primeras 5 filas):


,ano,departamento,desercion,desercion_transicion,desercion_primaria,desercion_secundaria,desercion_media,pobreza_monetaria,pobreza_extrema,total_sedes_urbanas,total_sedes_rurales,total_sedes,tasa_analfabetismo_rural_promedio,dato_numerico_h_promedio
0,2021,antioquia,4.81,3.59,4.27,5.90,4.13,32.8,9.2,NaN,NaN,NaN,NaN,NaN
1,2021,atlantico,1.67,2.15,1.75,1.68,1.03,42.1,12.3,NaN,NaN,NaN,NaN,NaN
2,2021,bogotadc,1.29,1.07,1.08,1.35,1.88,30.5,8.4,NaN,NaN,NaN,NaN,NaN
3,2021,bolivar,3.69,4.13,3.33,4.27,3.13,54.0,18.8,NaN,NaN,NaN,NaN,NaN
4,2021,boyaca,2.97,2.75,2.13,3.63,3.69,41.8,16.4,NaN,NaN,NaN,NaN,NaN


DataFrame Final consolidado (últimas 5 filas):


,ano,departamento,desercion,desercion_transicion,desercion_primaria,desercion_secundaria,desercion_media,pobreza_monetaria,pobreza_extrema,total_sedes_urbanas,total_sedes_rurales,total_sedes,tasa_analfabetismo_rural_promedio,dato_numerico_h_promedio
127,2024,amazonas,5.11,6.95,2.95,7.74,6.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
128,2024,guainia,4.73,4.84,3.87,6.69,3.71,NaN,NaN,NaN,NaN,NaN,NaN,NaN
129,2024,guaviare,4.8,6.83,3.93,6.23,2.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN
130,2024,vaupes,5.34,3.75,3.74,7.5,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
131,2024,vichada,6.31,4.05,6.26,7.53,4.83,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Dimensiones del DataFrame final: (132, 14)
Columnas del DataFrame final: ['ano', 'departamento', 'desercion', 'desercion_transicion', 'desercion_primaria', 'desercion_secundaria', 'desercion_media', 'pobreza_monetaria', 'pobreza_extrema', 'total_sedes_urbanas', 'total_sedes_rurales', 'total_sedes', 'tasa_analfabetismo_rural_promedio', 'dato_numerico_h_promedio']
